# Imports and setup

In [22]:
import json
import cohere

json_file = open('apikey.json')
data = json.load(json_file)

api_key = data['api_key']
co = cohere.ClientV2(api_key)

# a. folositi un LLM pre-antrenat (pe texte generale) si analizati influenta parametrilor (inclusiv a tokenizer-ului) asupra calitatii textului generat

In [23]:
def generate_prompt(first_lines):
    prompt = ("I have the first verses of each stanza of a poem. "
              "Please generate the rest of the poem, without adding more to the first verse."
              "Some verses are in english , some in romanian. "
              "The response should only contain the poem\n\n")
    
    for i,verse in enumerate(first_lines):
        prompt += f"Stanza {i+1}, First verse: {verse}\n"
    
    return prompt + "\n\n"

def generate_poem(first_lines, client):
    prompt = generate_prompt(first_lines)
    
    response = client.chat(
        model='command-a-03-2025',
        messages=[
            {
                'role': 'user',
                'content': prompt
            }
        ],
        temperature=0.7,
        max_tokens=1000,
    )
    
    return response.message.content[0].text 

In [24]:
first_lines = [
    "The water is cold",
    "The sun is shining",
    "The balls are big",
    "Apa trece, pietrele raman",
    "Europa , ne place sa o invadam!",
    "We love the way you talk to us",
    "The tree speaks with me in mystical ways",
    "I miss the old times i was young"
]

In [25]:
poem = generate_poem(first_lines, co)
print(poem)

**Stanza 1**  
The water is cold,  
Yet it holds the stories untold,  
A mirror to skies, a cradle of souls,  
Whispering secrets to those who are bold.  

**Stanza 2**  
The sun is shining,  
Its rays softly binding,  
The earth in its glow, all worries resigning,  
A promise of warmth, forever reminding.  

**Stanza 3**  
The balls are big,  
Rolling with a silent rig,  
Carrying weight, yet they never dig,  
A dance of fate, in a cosmic jig.  

**Stanza 4**  
Apa trece, pietrele raman,  
Timpul curge, amintirile stau,  
In valuri de vise, adevarul se ascunde,  
Pietrele tac, dar totul se spune.  

**Stanza 5**  
Europa, ne place sa o invadam!  
Cu ganduri de pace, cu vise sa o imbracam,  
In culorile ei, ne regasim, ne gasim,  
Un continent de vise, pe care il iubim.  

**Stanza 6**  
We love the way you talk to us,  
Your words a gentle, soothing husk,  
In every syllable, a world we trust,  
A melody that turns our hearts to dust.  

**Stanza 7**  
The tree speaks with me in mysti

# Influentza HyperParametrilor
# temperature => spune LLM cat de conservator (<1) sau haotic(>1) sa fie in generarea textului
# max_tokens => numarul maxim de token-uri generate , un cuvant poate sa aiba 2 sau 3 token-uri

# b. folositi un LLM pre-antrenat si adaptat la un corpus de poezii si analizati influenta parametrilor (inclusiv a tokenizer-ului) asupra calitatii textului generat

In [32]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained("prajwalcr/poetry_gpt2")
tokenizer = AutoTokenizer.from_pretrained("prajwalcr/poetry_gpt2")

def generate_poetry(prompt):
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids
    attention_mask = (input_ids != 0).long()
    output = model.generate(input_ids, 
                            do_sample=True, 
                            top_k=50, 
                            top_p=0.95, 
                            max_length=150,
                            attention_mask=attention_mask,
                            temperature=1.2,
                            pad_token_id=tokenizer.eos_token_id)
    
    return tokenizer.decode(output[0], skip_special_tokens=True)

In [33]:
for i,verse in enumerate(first_lines):
    print(f"Stanza {i+1}\n")
    print(generate_poetry(verse) + "\n")

Stanza 1
The water is cold like a wet sea;
It is almost a freeze;
A little under the moon,
Its soft touch is making my hair
cold.

Stanza 2
The sun is shining over the city
and you and I with silks and gold--
you and I--
as if on the silver moonlight
as I could be a sea-shell.

Stanza 3
The balls are big, he's as big in his biz,
And it is not fair to compare;
But then comes his little tricks,
The great hammer;
A gully; a bullock.

Stanza 4
Apa trece, pietrele ramanum, pietus, et pinguis
et tanti fraterum tuos pateret ante ducibus
haevis aereas deutscit pocas.

Stanza 5
Europa , ne place sa o invadam!
vis facies, parvula dicenis,
ut si ne vam plaudit ille
seruamque diuitare,
fraui, si nostra cernere.

Stanza 6
We love the way you talk to us, I see the difference in each
So often, our tongues cannot be so tense--
I cannot understand why you would speak so softly?
To talk with you makes them more like me, that.

Stanza 7
The tree speaks with me in mystical ways:
While thy magic chords wit

# Influentza HyperParametrilor
top_k=100 => alege aleator doar din top 100 cele mai bune token-uri
top_p=0.95 => in loc de limitare alege atatea token-uri cat probabiliate sa fie top_p%
max_length=200, => numarul maxim de tokeun-uri generate
attention_mask => masca de atentie pentru a ignora token-urile de padding
temperature => spune LLM cat de conservator (<1) sau haotic(>1) sa fie in generarea textului
pad_token_id=tokenizer.eos_token_id => dezactivare erori sau warning-uri


# c. Incercati sa raspundeti la urmatoarele intrebari:

## c.1 care sunt diferentele de calitate intre textele generate cu cele doua tipuri de LLM-uri?
   ####  Calitate primului LLM este afectata de calitate promptului , cat de detaliat este , iar al doilea are o performanta mai buna fiind antrenat pe un corpus de poezii.

## c.2 ce se intampla daca versurile din prompt sunt in limba engleza?
   #### Ambele LLm-uri sunt antrenate pe limba engleza, deci raspunsul dat este normal si in limba engleza.

## c.3 ce se intampla daca versurile din prompt sunt in limba romana?
   #### Primul LLM va genera un text in limba romana, cu conditia sa fie mentionata.

## c.4 ce se intampla daca versurile din prompt sunt in limba romana si corpusul de antrenare este in limba engleza?
   #### In LLM-uri pe corpus engleza vor incerca sa ghiceasca limba de cele mai multe ori va scrie in latina versuile.

## c.5 cum se poate "personaliza" LLM pentru a genera versuri in stil de pastel (cu accent pe frumusetea naturii)?
   #### Personalizare LLM-ului este data prin modificare promptului, adica sa fie mai detaliat si sa contina cuvinte cheie care sa sugereze natura , pastelul.